<a href="https://colab.research.google.com/github/Nandish4470/Analyzer_2.0/blob/main/analyzer_2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Google Colab Notebook - Elite Indian Railway Tender Parser v3.0
#
# STEP 1: Run this cell to install all required libraries.
!pip install -q pdfplumber tabula-py PyPDF2 pandas tqdm openpyxl
print("✓ All libraries installed.")

# ----------------------------------------------
# STEP 2: Define all Parsing Functions
# (Run this cell to load functions into memory)
# ----------------------------------------------
import os
import re
import io
import json
import pickle
import math
import time
import traceback
from collections import defaultdict, Counter
from tqdm import tqdm
from datetime import datetime
from decimal import Decimal, InvalidOperation

import pandas as pd
import numpy as np
import pdfplumber
import PyPDF2
try:
    import tabula
except Exception as e:
    print(f"Tabula-py might have issues: {e}")
    tabula = None

from google.colab import files
from IPython.display import Markdown, display, HTML

# --- Helper Functions ---

def to_number(s):
    """Robustly converts a string to a float, handling commas, (negatives), and N/A values."""
    if s is None:
        return np.nan
    if isinstance(s, (int, float, Decimal, np.integer, np.floating)):
        try:
            return float(s)
        except:
            return np.nan

    st = str(s).strip()
    if not st or st.lower() in ("na", "n/a", "-", "nan", "none", "nil"):
        return np.nan

    st = st.replace("\xa0", "").replace("$", "").replace("€", "").replace("£", "").replace("₹", "")

    neg = False
    if st.startswith("(") and st.endswith(")"):
        neg = True
        st = st[1:-1].strip()

    st = st.replace(",", "")
    # Remove any non-numeric characters except for decimal point, sign, and 'e' for scientific notation
    st_clean = re.sub(r"[^0-9eE\.\-]+", "", st)

    if st_clean in ("", ".", "-", "+"):
        return np.nan
    try:
        val = float(st_clean)
        return -val if neg else val
    except (ValueError, InvalidOperation):
        return np.nan

def normalize_text(text):
    """Cleans and normalizes text from PDF."""
    if not text:
        return ""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    # Consolidate all forms of whitespace and special PDF spaces
    text = re.sub(r"[\s\u00a0\u200b\u200c\u200d\ufeff]+", " ", text)
    return text.strip()

def confidence_msg(task, method, score):
    """Formats a confidence log message."""
    emoji = "✅" if score >= 85 else "⚠️" if score >= 70 else "❌"
    return f"{emoji} {task} → {method} | {score}% confidence"

def parse_nit_header_hybrid(page_text, page_tables):
    """
    Robust NIT Header parser.
    Uses a table-first approach (Method 1/2) and falls back to Regex (Method 3).
    """
    nit = {}

    # --- Table-based parsing (Priority) ---
    try:
        if page_tables:
            for table in page_tables:
                if not table or len(table) < 2: # Skip empty or single-row tables
                    continue

                # Check for 2-column key-value format
                if len(table[0]) == 2:
                    for row in table:
                        k = normalize_text(row[0])
                        v = normalize_text(row[1])
                        if k and v:
                            nit[k] = v

                # Check for 4 or 6 column key-value format
                elif len(table[0]) in (4, 6):
                    for row in table:
                        for i in range(0, len(row), 2):
                            k = normalize_text(row[i])
                            v = normalize_text(row[i+1])
                            if k and v:
                                nit[k] = v

    except Exception as e:
        pass # Fallback to regex

    # --- Regex-based parsing (Fallback) ---
    # Use raw page text for regex to catch multi-line values
    header_text_multiline = page_text.replace('\r','\n')

    # Define patterns for key-value pairs, ending at a newline
    patterns = {
        'Tender No': r'Tender No\s*[:\s]*\s*([A-Za-z0-9\-\_\/]+)',
        'Name of Work': r'Name of Work\s*[:\s]*\s*([\s\S]*?)(?=Bidding type)',
        'Bidding type': r'Bidding type\s*[:\s]*\s*([^\n]+)',
        'Tender Type': r'Tender Type\s*[:\s]*\s*([^\n]+)',
        'Bidding System': r'Bidding System\s*[:\s]*\s*([^\n]+)',
        'Tender Closing Date Time': r'Tender Closing Date\s+Time\s*[:\s]*\s*([^\n]+)',
        'Advertised Value': r'Advertised Value\s*[:\s]*\s*([\d\.,]+)',
        'Earnest Money (Rs.)': r'Earnest Money \(Rs\.\)\s*[:\s]*\s*([\d\.,]+)',
        'Period of Completion': r'Period of Completion\s*[:\s]*\s*([^\n]+)',
        'Contract Type': r'Contract Type\s*[:\s]*\s*([^\n]+)',
        'Are JV allowed to bid': r'Are JV allowed to bid\s*[:\s]*\s*([^\n]+)',
        'Bidding Start Date': r'Bidding Start Date\s*[:\s]*\s*([^\n]+)',
    }

    for k, pat in patterns.items():
        if k not in nit: # Only fill if table-parser missed it
            m = re.search(pat, header_text_multiline, re.IGNORECASE)
            if m:
                nit[k] = normalize_text(m.group(1)) # Clean up the match

    # --- Clean known messy fields ---
    if 'Name of Work' in nit:
        # Clean up common multi-line garbage
        nit['Name of Work'] = nit['Name of Work'].split('\n')[0].strip()

    # Final check for critical fields that regex might have missed
    if not nit.get('Advertised Value'):
        m = re.search(r'Advertised Value\s*([\d\.,]+)', header_text_multiline)
        if m: nit['Advertised Value'] = normalize_text(m.group(1))

    if not nit.get('Earnest Money (Rs.)') or to_number(nit.get('Earnest Money (Rs.)')) == 0:
        m = re.search(r'Earnest Money \(Rs\.\)\s*([\d\.,]+)', header_text_multiline)
        if m: nit['Earnest Money (Rs.)'] = normalize_text(m.group(1))

    return nit


def parse_schedule_header(line):
    """Identifies schedule headers, e.g., 'Item-1 Schedule A1 (2.0 Earthwork)'"""

    # Pattern: Item-1 Schedule A1 (2.0 Earthwork)
    m = re.search(r'Item-(\d+)\s+(Schedule\s+[A-Z0-9\s\(\)\.\-]+)', line, re.IGNORECASE)
    if m:
        # Standardize the key
        key = f"Schedule {m.group(2).split('Schedule')[-1].strip()}"
        key = re.sub(r'\(\)', '', key).strip() # Clean 'Schedule A1 ()'
        return key

    # Pattern: Schedule C-All NS items
    m = re.search(r'(Schedule\s*\(?_?\)?\s*[A-Z])-(All\s+(DSR|USSOR|NS)\s+items)', line, re.IGNORECASE)
    if m:
        return f"{m.group(1).strip()}-{m.group(2).strip()}"

    # Pattern: Schedule Schedule B-All USSOR-2021 Items
    m = re.search(r'Schedule\s+(Schedule\s+[A-Z].*Items)', line, re.IGNORECASE)
    if m:
        return m.group(1).strip()

    return None

def category_from_schedule(sched_key):
    """Standardizes schedule names into work categories."""
    sched_upper = sched_key.upper()

    # Try direct keyword matching
    if 'EARTHWORK' in sched_upper: return 'EARTHWORK'
    if 'R.C.C' in sched_upper or 'REINFORCED CEMENT CONCRETE' in sched_upper: return 'R.C.C WORK'
    if 'CONCRETE WORK' in sched_upper: return 'CONCRETE WORK'
    if 'STEEL WORK' in sched_upper: return 'STEEL WORK'
    if 'MASONARY' in sched_upper or 'MASONRY' in sched_upper: return 'MASONRY WORK'
    if 'FINISHING' in sched_upper: return 'FINISHING WORK'
    if 'WATER SUPPLY' in sched_upper: return 'WATER SUPPLY'
    if 'WATER PROOFING' in sched_upper: return 'WATER PROOFING'
    if 'DISMANTLING' in sched_upper: return 'DISMANTLING & DEMOLISHING'
    if 'FLOORING' in sched_upper: return 'FLOORING WORK'
    if 'ROOFING' in sched_upper: return 'ROOFING WORK'
    if 'USSOR' in sched_upper: return 'USSOR ITEMS'
    if 'NS ITEMS' in sched_upper: return 'NS ITEMS'

    # Fallback: Extract from parenthesis, cleaning out numbers
    m = re.search(r'\(([^)]+)\)', sched_key)
    if m:
        work_type = m.group(1).strip().upper()
        work_type = re.sub(r'^\d+\.\d+\s*', '', work_type) # Remove prefixes like '2.0 '
        if work_type: return work_type

    return sched_key # Fallback to the key itself

# --- State-Machine Item Parser (v3.0) ---

class TenderParserSM:
    """
    A state-machine parser to robustly extract items from tender documents.
    Handles multi-line descriptions and varying item/amount line positions.
    """
    def __init__(self):
        self.state = "SEEKING_SCHEDULE"
        self.current_schedule_key = "UNKNOWN"
        self.item_buffer = {}
        self.all_parsed_items = []
        self.log = []

        # REGEX PATTERNS
        # 1. Schedule Header: "Item-1 Schedule A1 (2.0 Earthwork)"
        self.RE_SCHED_HEADER = re.compile(r'Item-(\d+)\s+(Schedule\s+[A-Z0-9\s\(\)\.\-]+)', re.IGNORECASE)
        self.RE_SCHED_HEADER_ALT = re.compile(r'(Schedule\s*\(?_?\)?\s*[A-Z])-(All\s+(DSR|USSOR|NS)\s+items)', re.IGNORECASE)
        self.RE_SCHED_HEADER_ALT2 = re.compile(r'Schedule\s+(Schedule\s+[A-Z].*Items)', re.IGNORECASE)

        # 2. Item Start: "1 2.6.1 All kinds of soil"
        self.RE_ITEM_START = re.compile(r'^\s*(\d{1,3})?\s*([\d\.\-A-Z]+)\s+(.{10,})') # S.No (opt), Item No, Description Start

        # 3. Amount Line: "cum 800 205.45 328720"
        self.RE_AMOUNT_LINE = re.compile(r'(.+?)\s+([A-Za-z/%]{1,10})\s+([\d,\.\-]+)\s+([\d,\.\-]+)\s+([\d,\.\-eE\+]+)\s*$') # Desc_end, Unit, Qty, Rate, Amount

        # 4. Amount Line (alternative, where desc is already fully captured): "cum 800 205.45 328720"
        self.RE_AMOUNT_LINE_ONLY = re.compile(r'^\s*([A-Za-z/%]{1,10})\s+([\d,\.\-]+)\s+([\d,\.\-]+)\s+([\d,\.\-eE\+]+)\s*$') # Unit, Qty, Rate, Amount

    def parse_schedule_header(self, line):
        m = self.RE_SCHED_HEADER.match(line)
        if m:
            key = f"Schedule {m.group(2).split('Schedule')[-1].strip()}"
            key = re.sub(r'\(\)', '', key).strip()
            return key

        m = self.RE_SCHED_HEADER_ALT.match(line)
        if m:
            return f"{m.group(1).strip()}-{m.group(2).strip()}"

        m = self.RE_SCHED_HEADER_ALT2.match(line)
        if m:
            return m.group(1).strip()
        return None

    def commit_item(self):
        """Validates and saves the item in the buffer."""
        if not self.item_buffer:
            return

        # Join multi-line description
        if 'description_lines' in self.item_buffer:
            self.item_buffer['Description of Item'] = " ".join(self.item_buffer['description_lines'])
            del self.item_buffer['description_lines']

        # Validate
        if all(k in self.item_buffer for k in ['Item No', 'Description of Item', 'Unit', 'Qty', 'Rate', 'Amount']):
            qty = self.item_buffer['Qty']
            rate = self.item_buffer['Rate']
            amount = self.item_buffer['Amount']

            if pd.isna(qty) or pd.isna(rate) or pd.isna(amount):
                self.log.append(f"REJECT (Bad Num): {self.item_buffer.get('Item No')}")
            # Check for 10% tolerance, as some tenders have minor rounding
            elif math.isclose(qty * rate, amount, rel_tol=0.1):
                self.all_parsed_items.append(self.item_buffer)
            else:
                self.log.append(f"REJECT (Math Fail): {self.item_buffer.get('Item No')} | Q*R={qty*rate} vs A={amount}")
        else:
             self.log.append(f"REJECT (Missing Fields): {self.item_buffer.get('Item No')}")

        self.item_buffer = {} # Clear buffer after commit

    def feed_line(self, line, page_no):
        """Processes a single line of text from the PDF."""

        line_norm = normalize_text(line)
        if not line_norm or "Run Date/Time:" in line or "MUMBAI CENTRAL DIVISION" in line:
            return

        # --- STATE 1: CHECK FOR SCHEDULE HEADER ---
        new_schedule_key = self.parse_schedule_header(line_norm)
        if new_schedule_key:
            self.commit_item() # Commit any pending item from the previous schedule
            self.current_schedule_key = new_schedule_key
            self.state = "SEEKING_ITEM"
            return

        # --- STATE 2: CHECK FOR ITEM START ---
        # e.g., "1 2.6.1 All kinds of soil"
        item_start_match = self.RE_ITEM_START.match(line_norm)
        if item_start_match:
            self.commit_item() # Commit the previous item
            self.state = "ITEM_DESCRIPTION"
            self.item_buffer = {
                "S No.": item_start_match.group(1) or np.nan,
                "Item No": item_start_match.group(2).strip(),
                "description_lines": [item_start_match.group(3).strip()],
                "Page": page_no,
                "Schedule": self.current_schedule_key
            }
            return

        # --- STATE 3: CHECK FOR AMOUNT (if inside an item) ---
        if self.state == "ITEM_DESCRIPTION":

            # Case A: Amount is on the same line as the *end* of the description
            # e.g., "...as directed by Engineer-in-charge. cum 800 205.45 164360"
            amount_match = self.RE_AMOUNT_LINE.match(line_norm)
            if amount_match:
                self.item_buffer['description_lines'].append(amount_match.group(1).strip())
                self.item_buffer['Unit'] = amount_match.group(2)
                self.item_buffer['Qty'] = to_number(amount_match.group(3))
                self.item_buffer['Rate'] = to_number(amount_match.group(4))
                self.item_buffer['Amount'] = to_number(amount_match.group(5))
                self.commit_item()
                self.state = "SEEKING_ITEM" # Reset state
                return

            # Case B: Amount is on its *own* line
            # e.g., "cum 800 205.45 164360"
            amount_only_match = self.RE_AMOUNT_LINE_ONLY.match(line_norm)
            if amount_only_match:
                self.item_buffer['Unit'] = amount_only_match.group(1)
                self.item_buffer['Qty'] = to_number(amount_only_match.group(2))
                self.item_buffer['Rate'] = to_number(amount_only_match.group(3))
                self.item_buffer['Amount'] = to_number(amount_only_match.group(4))
                self.commit_item()
                self.state = "SEEKING_ITEM" # Reset state
                return

            # Case C: This is just another line of the description
            if not line_norm.startswith("Item-") and not line_norm.startswith("Schedule"):
                 self.item_buffer['description_lines'].append(line_norm)
                 return

    def get_results(self):
        self.commit_item() # Commit any final item

        item_breakup_dfs = {}
        if not self.all_parsed_items:
            return pd.DataFrame(), {}, [] # Return empty if nothing found

        all_items_df = pd.DataFrame(self.all_parsed_items)
        all_items_df['Category'] = all_items_df['Schedule'].apply(category_from_schedule)

        # Create individual DFs for each schedule
        for sched_key in all_items_df['Schedule'].unique():
            df = all_items_df[all_items_df['Schedule'] == sched_key].copy()
            # Reorder columns
            cols = ['S No.', 'Item No', 'Description of Item', 'Unit', 'Qty', 'Rate', 'Amount', 'Page', 'Schedule', 'Category']
            for c in cols:
                if c not in df.columns: df[c] = np.nan
            item_breakup_dfs[sched_key] = df[cols]

        return all_items_df, item_breakup_dfs, self.log


# --- Main Parsing Function ---

def parse_tender(pdf_path):

    start_time = time.time()
    progress_logs = []
    confidence_scores = {}

    results = {
        "nit_header": {},
        "schedules_summary_generated": pd.DataFrame(), # We will generate this
        "item_breakups": {},
        "eligibility_criteria": {"bullets": [], "raw_text": ""},
        "flags": [],
        "top10": {},
        "raw_text_pages": [],
        "all_items_df": pd.DataFrame(),
        "num_pages": 0,
        "parser_log": []
    }

    # --- Step 1: PDF Text & Table Extraction ---
    pages_text = []
    pages_tables = {}

    try:
        with pdfplumber.open(pdf_path) as pdf:
            results['num_pages'] = len(pdf.pages)
            progress_logs.append(f"Opened PDF with {results['num_pages']} pages.")

            for i, page in enumerate(tqdm(pdf.pages, desc="Extracting Pages", leave=False)):
                page_no = i + 1
                try:
                    text = page.extract_text(x_tolerance=2, y_tolerance=2) or ""
                    tables = page.extract_tables() or []
                    pages_text.append(text)
                    pages_tables[page_no] = tables
                except Exception as e:
                    progress_logs.append(f"Warning: Could not extract page {page_no}: {e}")
                    pages_text.append("")
                    pages_tables[page_no] = []

        results['raw_text_pages'] = pages_text
        progress_logs.append(confidence_msg("Text & Tables", "Method 2 (pdfplumber)", 98))
        confidence_scores['text_extraction'] = 98
    except Exception as e:
        progress_logs.append(f"FATAL: Could not open PDF: {e}")
        return results, progress_logs, confidence_scores

    # --- Step 2: NIT Header Parsing (Hybrid) ---
    try:
        page_1_text = pages_text[0]
        page_1_tables = pages_tables.get(1, [])
        results['nit_header'] = parse_nit_header_hybrid(page_1_text, page_1_tables)

        if 'Tender No' not in results['nit_header']:
            raise ValueError("Tender No not found")
        if to_number(results['nit_header'].get('Earnest Money (Rs.)')) is None or to_number(results['nit_header'].get('Earnest Money (Rs.)')) == 0:
            progress_logs.append(confidence_msg("NIT Header", "Hybrid Method", 60)) # Found, but key fields missing
            confidence_scores['nit_header'] = 60
        else:
            progress_logs.append(confidence_msg("NIT Header", "Hybrid Method", 95))
            confidence_scores['nit_header'] = 95

    except Exception as e:
        progress_logs.append(f"Error parsing NIT Header: {e}")
        confidence_scores['nit_header'] = 30

    # --- Step 3: Hierarchical Item Breakup Parsing (State Machine) ---
    try:
        sm = TenderParserSM()
        # Start scanning from page 1 (headers can be on page 1)
        for p in tqdm(range(results['num_pages']), desc="Parsing Items (SM)", leave=False):
            page_no = p + 1
            page_text = pages_text[p]
            lines = page_text.split('\n')
            for line in lines:
                sm.feed_line(line, page_no)

        # Get final results from the state machine
        all_items_df, item_breakup_dfs, parser_log = sm.get_results()

        results['all_items_df'] = all_items_df
        results['item_breakups'] = item_breakup_dfs
        results['parser_log'] = parser_log

        if all_items_df.empty:
            raise ValueError("State machine parser found no valid items.")

        progress_logs.append(confidence_msg("Item Breakups", "State Machine", 95))
        confidence_scores['item_breakups'] = 95

    except Exception as e:
        progress_logs.append(f"Item Breakup parsing error: {e}\n{traceback.format_exc()}")
        confidence_scores['item_breakups'] = 30

    # --- Step 4: Generate Schedule Summary (from parsed items) ---
    try:
        if not results['all_items_df'].empty:
            df = results['all_items_df']
            summary = df.groupby('Schedule')['Amount'].agg(['sum', 'count']).reset_index()
            summary.columns = ['Schedule', 'Total Amount', 'Item Count']
            summary = summary.sort_values('Total Amount', ascending=False)
            results['schedules_summary_generated'] = summary
            progress_logs.append(confidence_msg("Schedule Summary", "Generated from Items", 100))
            confidence_scores['schedule_summary'] = 100
        else:
            raise ValueError("all_items_df is empty, cannot generate summary.")
    except Exception as e:
        progress_logs.append(f"Schedule Summary generation error: {e}")
        confidence_scores['schedule_summary'] = 0

    # --- Step 5: Eligibility Criteria ---
    try:
        # Scan pages 4-35 (as per original logic)
        start_page = 4
        end_page = min(35, results['num_pages'])
        eligibility_text = "\n".join(pages_text[start_page:end_page])

        eligibility_bullets = []
        in_eligibility_section = False

        for line in eligibility_text.split('\n'):
            line_norm = normalize_text(line)
            line_upper = line_norm.UPPER()

            if 'ELIGIBILITY CONDITIONS' in line_upper or 'FINANCIAL CRITERIA' in line_upper or 'TECHNICAL CRITERIA' in line_upper:
                in_eligibility_section = True

            if not in_eligibility_section:
                continue

            # Look for bullet points or numbered lists
            if re.match(r'^\s*(\d+\.\d+|\(i+\)|\([a-z]\)|[a-z]\)|\d+\))\s', line_norm, re.IGNORECASE) or line_norm.startswith('•'):
                if len(line_norm.split()) > 5: # Filter out simple headings
                    eligibility_bullets.append(line_norm)

        results['eligibility_criteria']['bullets'] = eligibility_bullets
        results['eligibility_criteria']['raw_text'] = eligibility_text

        progress_logs.append(confidence_msg("Eligibility Criteria", "Keyword Scan", 80 if eligibility_bullets else 45))
        confidence_scores['eligibility'] = 80 if eligibility_bullets else 45

    except Exception as e:
        progress_logs.append(f"Eligibility parsing error: {e}")
        confidence_scores['eligibility'] = 30

    # --- Step 6: Top 10 Cost Drivers (Hybrid Logic) ---
    try:
        if not results['all_items_df'].empty:
            df = results['all_items_df']
            df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce').fillna(0.0)

            # 1. Process Schedule A
            df_sched_A = df[df['Schedule'].str.contains('Schedule A', case=False, na=False)]
            sched_A_groups = df_sched_A.groupby('Category')['Amount'].sum().reset_index()
            sched_A_groups.rename(columns={'Category': 'Rankable Entity'}, inplace=True)
            sched_A_groups['Type'] = 'Group (Sch A)'

            # 2. Process Other Schedules
            df_other_scheds = df[~df['Schedule'].str.contains('Schedule A', case=False, na=False)]
            # We rank by *individual item* for other schedules
            other_items = df_other_scheds[['Description of Item', 'Amount']].copy()
            other_items.rename(columns={'Description of Item': 'Rankable Entity'}, inplace=True)
            other_items['Type'] = 'Item'

            # 3. Combine and Rank
            combined_ranking = pd.concat([sched_A_groups, other_items], ignore_index=True)
            combined_ranking = combined_ranking.sort_values('Amount', ascending=False)

            total_amt = combined_ranking['Amount'].sum()
            combined_ranking['Pct'] = (combined_ranking['Amount'] / total_amt * 100) if total_amt > 0 else 0

            results['top10'] = {
                'total_amount': total_amt,
                'drivers': combined_ranking.head(10).to_dict('records')
            }
            progress_logs.append(confidence_msg("Top 10 Drivers", "Hybrid Aggregation", 95))
            confidence_scores['top10'] = 95
        else:
            raise ValueError("all_items_df is empty, cannot calculate Top 10.")

    except Exception as e:
        progress_logs.append(f"Top 10 Drivers error: {e}")
        confidence_scores['top10'] = 20

    # --- Step 7: Flags ---
    try:
        header_flags = results['nit_header']
        if 'No' in header_flags.get('Are JV allowed to bid', 'Yes').upper():
            results['flags'].append("JV NOT ALLOWED")

        if 'Single Packet' in header_flags.get('Bidding System', ''):
            results['flags'].append("Single Packet System")

        em = to_number(header_flags.get('Earnest Money (Rs.)'))
        if em is not None and em > 0:
            results['flags'].append(f"Earnest Money: ₹{em/1_00_000:.2f} Lakh")

        progress_logs.append("✓ Flags identified.")
    except Exception as e:
        progress_logs.append(f"Flags error: {e}")

    end_time = time.time()
    results['parse_time_seconds'] = end_time - start_time
    progress_logs.append(f"Total processing time: {results['parse_time_seconds']:.2f} seconds.")

    return results, progress_logs, confidence_scores

# ----------------------------------------------
# STEP 3: Upload, Execute Parser, and Display Results
# ----------------------------------------------

# --- 3a. Upload File ---
print("Please upload your tender PDF:")
uploaded = files.upload()
pdf_filename = None

if uploaded:
    pdf_filename = next(iter(uploaded))
    print(f"\n✓ Uploaded '{pdf_filename}'")

    # --- 3b. Run the Parser ---
    print(f"--- Analyzing '{pdf_filename}' ---")
    parsed_data, logs, confidences = parse_tender(pdf_filename)

    # --- 3c. Format the Output ---

    # Header
    tender_no = parsed_data['nit_header'].get('Tender No', 'N/A')
    try:
        adv_val_str = f"₹{to_number(parsed_data['nit_header'].get('Advertised Value')) / 1_00_00_000:.2f} Cr"
    except:
        adv_val_str = f"₹{parsed_data['nit_header'].get('Advertised Value', 'N/A')}"

    name_of_work = parsed_data['nit_header'].get('Name of Work', 'N/A')

    md = f"# TENDER {tender_no} | {name_of_work} | {adv_val_str}\n\n"

    # Flags
    if parsed_data['flags']:
        flags_html = " ".join([f"<span style='color:red; font-weight:bold; border: 1px solid red; padding: 2px 5px; border-radius: 5px; margin-right: 10px;'>{flag}</span>" for flag in parsed_data['flags']])
        md += f"{flags_html}\n\n"

    # NIT Header Table
    md += "## 📄 NIT HEADER\n\n"
    if parsed_data['nit_header']:
        header_df = pd.DataFrame(list(parsed_data['nit_header'].items()), columns=['Field', 'Value'])
        md += header_df.to_markdown(index=False) + "\n\n"
    else:
        md += "❌ Could not extract NIT header data.\n\n"

    # Schedule Summary (GENERATED)
    md += "## SCHEDULE SUMMARY (Generated from Item Breakup)\n\n"
    if not parsed_data['schedules_summary_generated'].empty:
        summary_df = parsed_data['schedules_summary_generated'].copy()
        summary_df['Total Amount'] = summary_df['Total Amount'].apply(lambda x: f"₹ {x:,.2f}")
        md += summary_df.to_markdown(index=False) + "\n\n"
    else:
        md += "❌ No item data found, could not generate schedule summary.\n\n"

    # Top 10 Drivers
    md += "## 🎯 TOP 10 COST DRIVERS (Global)\n\n"
    if parsed_data['top10'] and parsed_data['top10'].get('drivers'):
        total = parsed_data['top10']['total_amount']
        md += f"**Total Estimated Value (from parsed items)**: ₹ {total:,.2f}\n\n"

        top10_df = pd.DataFrame(parsed_data['top10']['drivers'])
        top10_df['Amount'] = top10_df['Amount'].apply(lambda x: f"₹ {x:,.2f}")
        top10_df['Pct'] = top10_df['Pct'].apply(lambda x: f"{x:.1f}%")
        top10_df.columns = ['Rankable Entity', 'Total Amount', 'Type', 'Percentage']

        md += top10_df.to_markdown(index=False) + "\n\n"
    else:
        md += "❌ Could not compute cost drivers. Item parsing may have failed.\n\n"

    # Eligibility Criteria
    md += "## ✅ ELIGIBILITY CRITERIA\n\n"
    if parsed_data['eligibility_criteria']['bullets']:
        md += "*(Top 10 extracted criteria)*\n"
        for i, bullet in enumerate(parsed_data['eligibility_criteria']['bullets'][:10]):
            md += f"• {bullet}\n"
    else:
        md += "❌ No specific eligibility criteria bullets found in text scan.\n\n"

    # Full Item Breakup
    md += "## 📊 FULL ITEM BREAKUP (Sample)\n\n"
    if parsed_data['item_breakups']:
        md += f"Found {len(parsed_data['item_breakups'])} schedules and {len(parsed_data['all_items_df'])} total items.\n\n"

        # Sort schedules by total amount
        sorted_schedules = sorted(
            parsed_data['item_breakups'].items(),
            key=lambda item: item[1]['Amount'].sum(),
            reverse=True
        )

        for sched_key, df in sorted_schedules:
            md += f"### {sched_key}\n"
            md += f"**Total: ₹{df['Amount'].sum():,.2f}** ({len(df)} items)\n\n"

            # Display sample (first 5)
            sample_df = df.head(5).copy()
            sample_df['Amount'] = sample_df['Amount'].apply(lambda x: f"{x:,.2f}")
            sample_df['Rate'] = sample_df['Rate'].apply(lambda x: f"{x:,.2f}")

            md += sample_df[['S No.', 'Item No', 'Description of Item', 'Unit', 'Qty', 'Rate', 'Amount']].to_markdown(index=False) + "\n"
            if len(df) > 5:
                md += f"*...and {len(df) - 5} more items.*\n"
            md += "\n---\n"

    else:
        md += "❌ No item breakups extracted.\n\n"

    # --- Display Output ---
    display(Markdown(md))

    # --- Display Logs ---
    print("\n" + "="*60)
    print("📋 PARSING LOGS")
    print("="*60)
    for log in logs:
        print(log)
    if parsed_data['parser_log']:
        print("\n--- Item Parser Rejection Log ---")
        for log in parsed_data['parser_log'][:10]: # Print first 10 rejections
            print(log)
        if len(parsed_data['parser_log']) > 10:
            print(f"...and {len(parsed_data['parser_log']) - 10} more rejections.")

    # --- Export Buttons ---
    print("\n" + "="*60)
    print("📦 EXPORT OPTIONS")
    print("="*60)

    try:
        # 1. Schedule Amounts Summary
        if not parsed_data['schedules_summary_generated'].empty:
            parsed_data['schedules_summary_generated'].to_csv('schedule_amounts.csv', index=False)
            files.download('schedule_amounts.csv')
            print("✓ Downloaded: schedule_amounts.csv")
    except Exception as e:
        print(f"✗ Export error (schedule_amounts): {e}")

    try:
        # 2. Full Breakup
        if not parsed_data['all_items_df'].empty:
            with pd.ExcelWriter('full_breakup.xlsx') as writer:
                # Write all items to one sheet
                parsed_data['all_items_df'].to_excel(writer, sheet_name='All_Items', index=False)
                # Write each schedule to its own sheet
                for sched_key, df in parsed_data['item_breakups'].items():
                    safe_sheet_name = re.sub(r'[\r\n\t\[\]\*\:\?\/]', '', sched_key)[:30] # Clean sheet name
                    df.to_excel(writer, sheet_name=safe_sheet_name, index=False)

            files.download('full_breakup.xlsx')
            print("✓ Downloaded: full_breakup.xlsx (includes all items + individual schedule tabs)")
    except Exception as e:
        print(f"✗ Export error (full_breakup): {e}")

else:
    print("⚠️ No file uploaded. Please run this cell again to upload your tender PDF.")

SyntaxError: invalid decimal literal (ipython-input-3994269427.py, line 692)